In [ ]:
import pandas as pd
import sqlite3

df_raw = pd.read_csv("/content/ai_job_market_snapshot.csv")

# df_raw.head(10)

conn = sqlite3.connect("job_market.db")
df_raw.to_sql("raw_job_market", conn, if_exists="replace", index=False);

In [ ]:
# Inspect the actual columns in the DataFrame to find the correct experience column name
print("Available columns:", df_raw.columns.tolist())

# Dynamically select the correct experience column (e.g., 'Experience_Level' or similar)
exp_col = [col for col in df_raw.columns if 'exp' in col.lower()]
exp_column_name = exp_col[0] if exp_col else 'Experience'

sql_query = f"""
SELECT
      Job_ID,
      LOWER(TRIM(Industry)) As raw_industry,
      Role,
      {exp_column_name} As Experience,
      Employment_Type,
      Remote_Work_Model,
      AI_Adoption_Level,
      Automation_Risk_Score,
      Salary_USD,
      Skills_Required,
      Market_Demand_Index
FROM raw_job_market
WHERE Employment_Type = 'Full-time'
"""

df_extracted = pd.read_sql_query(sql_query, conn)
print(f"Extracted {df_extracted.shape[0]} robust corporate profiles for deep-dive analysis.")

In [ ]:
import numpy as np

df_extracted['Industry'] = df_extracted['raw_industry'].str.strip().str.title()
df_extracted.drop(columns=['raw_industry'],inplace=True)


In [ ]:
duplicate_count = df_extracted.duplicated(subset=['Job_ID']).sum()

In [ ]:
df_extracted = df_extracted.drop_duplicates(subset=['Job_ID'],keep='first')

In [ ]:
print(f"Duduplication complete. Removed {duplicate_count} data line anomalies.")


In [ ]:
df_extracted['Salary_USD'] = df_extracted.groupby(['Industry','Experience'])['Salary_USD'].transform(lambda c:c.fillna(c.median()))

In [ ]:
df_extracted['Remove_Work_Model'] = df_extracted['Remote_Work_Model'].fillna("Not Disclosed")
df_extracted['Automation_Risk_Score'] = df_extracted['Automation_Risk_Score'].fillna(df_extracted['Automation_Risk_Score'].median())
df_extracted['Skills_Required'] = df_extracted['Skills_Required'].fillna("Not Disclosed")

print("Misssing data engineering pass completed. Structural Voids resolved.")

In [ ]:
# Module implmentation

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

In [ ]:
df_extracted.columns

In [ ]:

x = df_extracted[['Experience','Remote_Work_Model',
       'AI_Adoption_Level',
       'Industry']]
y = df_extracted['Automation_Risk_Score']

In [ ]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size = 0.2, random_state=42)

In [ ]:
categorical_features = ['Experience','Remote_Work_Model','AI_Adoption_Level','Industry']

In [ ]:
categorical_transformer = OneHotEncoder(handle_unknown ='ignore',sparse_output=False)

In [ ]:
preprocessor = ColumnTransformer(transformers = [('cat', categorical_transformer, categorical_features)])

In [ ]:
ml_pipeline = Pipeline(steps=[('preprocessor',preprocessor),('regressor',RandomForestRegressor(n_estimators=100, random_state =42))])

In [ ]:
ml_pipeline.fit(x_train, y_train)

In [ ]:
y_pred = ml_pipeline.predict(x_test)

mae = mean_absolute_error(y_test, y_pred)

r2 = r2_score(y_test, y_pred)

In [ ]:
print(f"Machine Learning Model Metrics Evaluation:")
print(f"Mean Absolute Error : {mae:.4f} (Minimal deviation on risk indexing)")
print(f"R2 Variance Score: {r2:.4f}")



In [ ]:
df_extracted.to_csv("clean_AI_job_market.csv", index=False)

In [ ]:
df = pd.read_csv("/content/clean_AI_job_market.csv")
df.head(10)